# 12. Choosing a variant

The whole point of a batched library is that many small problems are solved in
one launch. This notebook measures that, and shows how to pick between the
`syev` variants for a given $(n, \text{batch})$ instead of guessing.

Timings include the host round trip (NumPy in, NumPy out), which is what you
actually pay from Python. Absolute numbers are machine-specific — the
*relationships* are the interesting part.

In [1]:
import numpy as np

import batchlas as bl

from _common import (
    batched_symmetric,
    eigenvalue_error,
    header,
    preferred_device,
    report,
    section,
    timed,
)

header("12. Choosing a variant")

device = preferred_device()
report("device", device or "library default")


12. Choosing a variant
          device: gpu


## One launch beats a Python loop

The arithmetic is identical either way. The difference is launch overhead and
device occupancy — and it is large.

In [2]:
section("One launch beats a Python loop")

batch, n = 512, 16
matrices = batched_symmetric(batch, n, seed=1)

_, batched_time = timed(lambda: bl.syev_cta(matrices, compute_vectors=False, device=device))


def one_at_a_time():
    return [bl.syev_cta(matrices[i], compute_vectors=False, device=device) for i in range(batch)]


_, looped_time = timed(one_at_a_time, repeats=1)

report(f"batched  ({batch} x {n}x{n})", f"{batched_time * 1e3:.2f} ms")
report(f"looped   ({batch} calls)", f"{looped_time * 1e3:.2f} ms")
report("speed-up", f"{looped_time / batched_time:.1f}x")


-- One launch beats a Python loop


          batched  (512 x 16x16): 2.93 ms
          looped   (512 calls): 467.95 ms
          speed-up: 159.8x


## How throughput scales with batch size

Cost per matrix falls steeply as the batch grows: a single small matrix is
dominated by fixed overhead, while a large batch amortises it away.

In [3]:
section("How throughput scales with batch size")

for batch_size in (1, 16, 128, 1024):
    matrices = batched_symmetric(batch_size, 16, seed=2)
    _, elapsed = timed(lambda: bl.syev_cta(matrices, compute_vectors=False, device=device))
    report(
        f"batch={batch_size:<5d}",
        f"{elapsed * 1e3:7.2f} ms total, {elapsed / batch_size * 1e6:7.1f} us per matrix",
    )


-- How throughput scales with batch size
          batch=1    :    0.95 ms total,   954.0 us per matrix
          batch=16   :    1.34 ms total,    83.7 us per matrix
          batch=128  :    1.53 ms total,    11.9 us per matrix
          batch=1024 :    4.50 ms total,     4.4 us per matrix


## Which `syev` variant wins at which size?

`syev_cta` and `syev_jacobi_cta` only accept $n \le 32$; the blocked and
two-stage paths are built for larger $n$. Rather than assume, ask the device
with `syev_variant_support` and then measure.

Eigenvalue error is reported alongside the timing so a fast-but-wrong variant
cannot hide.

In [4]:
section("Which syev variant wins at which size?")

for n, batch_size in ((16, 512), (32, 256), (128, 32), (512, 4)):
    matrices = batched_symmetric(batch_size, n, seed=3)
    reference = np.linalg.eigvalsh(matrices)
    support = bl.syev_variant_support(matrices, device=device)

    candidates = ["syev"]
    if n <= 32 and support["cta"]:
        candidates += ["syev_cta", "syev_jacobi_cta"]
    if support["blocked"]:
        candidates.append("syev_blocked")
    if support["two_stage"]:
        candidates.append("syev_two_stage")

    report(f"n={n}, batch={batch_size}", "")
    for name in candidates:
        try:
            values, elapsed = timed(
                lambda fn=name: getattr(bl, fn)(matrices, compute_vectors=False, device=device)
            )
            error = eigenvalue_error(values, reference)
            report(f"  {name:16s}", f"{elapsed * 1e3:8.2f} ms   max eigenvalue error {error:.2e}")
        except (RuntimeError, NotImplementedError) as exc:
            report(f"  {name:16s}", f"unavailable ({type(exc).__name__})")


-- Which syev variant wins at which size?
          n=16, batch=512: 
            syev            :     2.58 ms   max eigenvalue error 2.04e-14
            syev_cta        :     2.50 ms   max eigenvalue error 2.04e-14
            syev_jacobi_cta :     1.76 ms   max eigenvalue error 3.29e-14
            syev_blocked    :     2.82 ms   max eigenvalue error 1.78e-14


            syev_two_stage  :    19.11 ms   max eigenvalue error 2.13e-14
          n=32, batch=256: 


            syev            :     5.00 ms   max eigenvalue error 3.38e-14
            syev_cta        :     6.76 ms   max eigenvalue error 3.38e-14
            syev_jacobi_cta :     4.63 ms   max eigenvalue error 7.46e-14
            syev_blocked    :     6.05 ms   max eigenvalue error 2.93e-14
            syev_two_stage  :    15.37 ms   max eigenvalue error 4.09e-14
          n=128, batch=32: 


            syev            :    18.80 ms   max eigenvalue error 8.88e-14
            syev_blocked    :    18.73 ms   max eigenvalue error 8.88e-14
            syev_two_stage  :    36.39 ms   max eigenvalue error 9.41e-14


          n=512, batch=4: 


            syev            :   105.80 ms   max eigenvalue error 2.52e-13


            syev_blocked    :   100.94 ms   max eigenvalue error 2.52e-13
            syev_two_stage  :   121.41 ms   max eigenvalue error 2.56e-13


## CPU versus GPU

Small batches often do not justify a device transfer; large ones clearly do.

Note that `syev_cta` and the other `*_cta` routines need a sub-group width of
32 and are therefore GPU-only. The general `syev` driver runs on both, so it is
what this comparison uses.

In [5]:
section("CPU versus GPU for the same call")

for batch_size in (1, 256):
    matrices = batched_symmetric(batch_size, 16, seed=4)
    line = []
    for target in ("cpu", "gpu"):
        try:
            _, elapsed = timed(lambda t=target: bl.syev(matrices, compute_vectors=False, device=t))
            line.append(f"{target}: {elapsed * 1e3:8.2f} ms")
        except (RuntimeError, NotImplementedError) as exc:
            line.append(f"{target}: unavailable ({type(exc).__name__})")
    report(f"batch={batch_size:<5d}", "   ".join(line))


-- CPU versus GPU for the same call
          batch=1    : cpu:     0.21 ms   gpu:     0.65 ms
          batch=256  : cpu:     5.76 ms   gpu:     1.65 ms


## Two easy savings

Skipping the eigenvectors avoids the back-transform entirely, and `float32`
halves the memory traffic.

In [6]:
section("Skipping eigenvectors is a real saving")

matrices = batched_symmetric(256, 32, seed=5)
_, with_vectors = timed(lambda: bl.syev_cta(matrices, compute_vectors=True, device=device))
_, values_only = timed(lambda: bl.syev_cta(matrices, compute_vectors=False, device=device))

report("with eigenvectors", f"{with_vectors * 1e3:.2f} ms")
report("values only", f"{values_only * 1e3:.2f} ms")

section("float32 versus float64")

matrices = batched_symmetric(512, 16, seed=6)
for dtype in (np.float32, np.float64):
    typed = matrices.astype(dtype)
    _, elapsed = timed(lambda m=typed: bl.syev_cta(m, compute_vectors=False, device=device))
    report(f"{np.dtype(dtype).name}", f"{elapsed * 1e3:.2f} ms")


-- Skipping eigenvectors is a real saving
          with eigenvectors: 4.43 ms
          values only: 3.37 ms

-- float32 versus float64
          float32: 0.86 ms
          float64: 2.02 ms
